# Connections Solver — walkthrough
Live tour of the pipeline for the TA check-in. Each section runs the real code in `src/`.
Run from the repo root folder (the first cell handles the path).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from src import data
from src.data import read_raw, build_puzzles, split, pretty, RAW_DIR

## Step 1 — The shared data shape (`src/puzzle.py`)
Every module speaks the same two types, so Kevin (data, scorers, search) and Emme (game, policy, LLM, eval)
can work in parallel. Both are **frozen dataclasses**: immutable (no accidental edits mid-experiment) and
hashable (usable as cache keys — the scorers cache embeddings per puzzle).

In [ ]:
from src.puzzle import Puzzle, Group, LEVELS
g = Group(words=("BASS", "PIKE", "SOLE", "TROUT"), name="FISH", level=0)
print(g)
print("level 0 =", LEVELS[g.level])

In [ ]:
try:
    g.level = 3
except Exception as e:
    print("frozen ->", type(e).__name__)
cache = {g.words: "cached embeddings would live here"}   # hashable -> works as a dict key
cache[g.words]

## Step 2 — Data: load, clean, split
**Source:** `eric27n/NYT-Connections` on Hugging Face, **one row per word** (16 rows per game).

In [ ]:
raw = read_raw()          # downloads once into data/raw/, then cached
print(raw.shape)
raw.head(16)              # one full game

### The NULL trap
By default pandas converts the *text* `NULL`, `NA`, `NAN`, `NONE` into missing values — but those are real
Connections answers. Compare default loading vs. ours (everything read as plain text, auto-NaN off):

In [ ]:
csv = sorted(RAW_DIR.glob("*.csv"))[0]
naive = pd.read_csv(csv)
lost = naive[naive["Word"].isna()].index
print(f"default pandas: {len(lost)} words became NaN")
pd.DataFrame({"default pandas": naive.loc[lost, "Word"], "our loader": raw.loc[lost, "word"],
              "game": raw.loc[lost, "game_id"], "group": raw.loc[lost, "group_name"]})

### Group rows → `Puzzle` objects, drop invalid ones
Keep a game only if: 16 unique non-empty words, 4 groups of 4, levels exactly {0,1,2,3}.

In [ ]:
puzzles = build_puzzles(raw)   # prints kept / dropped counts and reasons

### Chronological split
Test = newest 100 (least likely in LLM training data), val = 50 before that, train = rest.
Never train on the future and test on the past; settings are tuned on **val only**.

In [ ]:
train, val, test = split(puzzles)
pd.DataFrame([{"split": n, "puzzles": len(ps), "from": ps[0].date, "to": ps[-1].date}
              for n, ps in [("train", train), ("val", val), ("test", test)]])

### Sanity check by eye
One puzzle as a board — check it against the NYT Connections archive for that date.

In [ ]:
print(pretty(test[0]))